# Inferencing using RDFS and OWL-RL — User Guide


Inferencing is the process of deriving new information from existing graph information according to a specified set of rules, semantics, or expressions.  Inferencing creates new triples based on existing triples in the graph.  

StarLayer supports inferencing in a variety of areas. 
- rdfs/owl-rl inferencing (this guide)
- owl 2 dl reasoning via HermiT ([OWL 2 DL Reasoning guide](05e-owl-dl-reasoning.ipynb) - a separate guide, since it needs a real Java runtime, not just a pip install)
- shacl rules ([SHACL inference rules](04b-shacl-inference-rules.ipynb))
- sparql rules (pending)

StarLayer also supports reasoning during queries — see the [SPARQL inferencing guide](03b-sparql-inferencing.ipynb).

Owlrl is a python package that implements rdfs/owl-rl reasoning. StarLayer integrates owl-rl via `StarLayerGraph.infer()`. 

This guide provides a brief overview of inferencing with StarLayerGraph, via `infer()`.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later sections may reuse variables from earlier sections.

In [1]:
from starlayer import StarLayerGraph, Namespace, RDF

EX = Namespace("http://example.org/")

def show(graph, label="Resulting graph", also_ns=None):
    """Helper function to serialize a graph as Turtle 1.2 - used throughout this
    guide to show a graph's state before/after inferencing.

    Filtered to ex: subjects only: infer() also adds a series of RDFS/OWL/XSD
    axiomatic vocabulary triples (rdfs:Resource typing, owl:sameAs reflexivity
    for every built-in class/datatype, etc.) Excluding these triples avoids
    unneeded complexity to the example output.

    also_ns: an additional namespace to include, matched against the predicate or
    object - useful for a triple whose subject isn't ex: but is still worth showing
    (e.g. an owlrl inconsistency triple, whose subject is a synthetic BNode).

    run print(g.serialize(format="turtle12")) to see the complete graph.
    """
    filtered = StarLayerGraph()
    filtered.bind("ex", EX)
    if also_ns is not None:
        filtered.bind("err", also_ns)
    for s, p, o in graph:
        also = also_ns is not None and (str(p).startswith(str(also_ns)) or str(o).startswith(str(also_ns)))
        if str(s).startswith(str(EX)) or also:
            filtered.add((s, p, o))
    print(f"\n{label}")
    print(filtered.serialize(format="turtle12"))

## How `infer()` works

`StarLayerGraph.infer(profile=..., mode=...)` has two independent parameters:

- **`profile`** — controls which ruleset to use in inferencing: `"rdfs"`, `"owl-rl"`, or `"rdfs+owl-rl"` (both).
- **`mode`** — controls the output: `"full"` (default, a **new** graph with the original triples plus everything entailed, original untouched), `"delta"` (a **new** graph with just the newly-entailed triples), or `"in-place"` (mutated original graph). All three ultimately reach the same *closure* — the deductive-closure concept, original data plus everything entailed from it — just in a different place and shape.

Note: with `mode="in-place"`, subsequent changes to the graph do not delete any entailed triples an earlier `infer()` call already added.

## 1. RDFS reasoning

`g.infer(profile="rdfs")` or `g.infer(profile="rdfs", mode="full")` materializes RDFS entailments into a new graph. The example below shows `rdfs:subClassOf` transitivity — `g` itself is left untouched.

In [2]:
# alice is a manager which means she is also an employee
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

show(g, "Before")

closed = g.infer(profile="rdfs")

show(closed, "After infer(profile='rdfs')")
print("original graph untouched:", len(g), "triples")


Before
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee .

ex:alice a ex:Manager .


After infer(profile='rdfs')
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a rdfs:Resource, ex:Employee, ex:Manager .

original graph untouched: 2 triples


## 2. OWL 2 RL reasoning

`g.infer(profile="owl-rl")` runs a similar forward-chaining closure, but over the OWL 2 RL rule set, producing entailments RDFS alone does not, such as those from `owl:equivalentClass` and property characteristics like `owl:TransitiveProperty`.

### 2.1 `owl:equivalentClass`

Two systems label the same role differently — HR calls it `ex:Manager`, the CRM calls the same thing `ex:TeamLead`. Declaring them `owl:equivalentClass` lets an instance of one be recognized as the other.

In [3]:
# alice is a manager, but she is also a teamlead, because manager and teamlead are the same
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
    ex:Manager rdfs:subClassOf ex:Employee .

""", format="turtle")

show(g, "Before")

closed = g.infer(profile="owl-rl")

show(closed, "After infer(profile='owl-rl')")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:alice a ex:Manager .


After infer(profile='owl-rl')
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee owl:sameAs ex:Employee .

ex:Manager rdfs:subClassOf ex:TeamLead, ex:Employee ;
    owl:equivalentClass ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:TeamLead rdfs:subClassOf ex:Manager, ex:Employee ;
    owl:equivalentClass ex:Manager ;
    owl:sameAs ex:TeamLead .

ex:alice a ex:Employee, ex:TeamLead, ex:Manager ;
    owl:sameAs ex:alice .



### 2.2 `owl:TransitiveProperty`

Declaring `ex:partOf` to be an `owl:TransitiveProperty` lets a chain of `partOf` facts entail the full transitive closure, not just each direct link.

In [4]:
# the SalesTeam is part of the SalesDepartment, which means the SalesTeam is a part of AcmeCorp
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .
""", format="turtle")

show(g, "Before")

closed = g.infer(profile="owl-rl")

show(closed, "After infer(profile='owl-rl')")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:partOf a owl:TransitiveProperty .


After infer(profile='owl-rl')
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:AcmeCorp owl:sameAs ex:AcmeCorp .

ex:SalesDept ex:partOf ex:AcmeCorp ;
    owl:sameAs ex:SalesDept .

ex:SalesTeam ex:partOf ex:SalesDept, ex:AcmeCorp ;
    owl:sameAs ex:SalesTeam .

ex:partOf a owl:TransitiveProperty ;
    owl:sameAs ex:partOf .



### 2.3 Running RDFS and OWL 2 RL together

`g.infer(profile="rdfs+owl-rl")` combines both rule sets into a single closure. What `"rdfs+owl-rl"` mostly adds on top of `"owl-rl"` alone is RDFS's *vocabulary-level* rules — universal `rdfs:Resource` typing for every term, visible below as `a owl:Thing, rdfs:Resource` on every `ex:` subject, absent from a `profile="owl-rl"`-only closure of the same data.

In [5]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .
""", format="turtle")

show(g, "Before")

closed = g.infer(profile="rdfs+owl-rl")

show(closed, "After infer(profile='rdfs+owl-rl')")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:alice a ex:Manager .

ex:partOf a owl:TransitiveProperty .


After infer(profile='rdfs+owl-rl')
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:AcmeCorp a rdfs:Resource, owl:Thing ;
    owl:sameAs ex:AcmeCorp .

ex:Employee a owl:Thing, rdfs:Resource ;
    owl:sameAs ex:Employee .

ex:Manager a rdfs:Resource, owl:Thing ;
    rdfs:subClassOf ex:Manager, ex:Employee, ex:TeamLead ;
    owl:equivalentClass ex:TeamLead, ex:Manager ;
    owl:sameAs ex:Manager .

ex:SalesDept ex:partOf ex:AcmeCorp ;
    a rdfs:Resource, owl:Thing ;
   

### 2.4 Detecting inconsistencies

`owlrl` also checks a number of the OWL 2 RL "consistency" rules while it reasons — e.g. two classes declared `owl:disjointWith` sharing a member, `owl:sameAs` clashing with `owl:differentFrom`, or a cardinality restriction being violated. It doesn't raise an exception when one is found; instead it adds an ordinary triple into the result graph. Checking for `err:error` triples in the result is how you detect inconsistencies.

Not everything is checked this way: literal-*value* distinctness (e.g. an `owl:FunctionalProperty` given two different literal values for the same subject) is explicitly unimplemented in the current version of `owlrl` — its own source comments this as "skipped on purpose at the moment" — so that kind of inconsistency passes through silently rather than being flagged.

In [6]:
# Dog and Cat are declared disjoint, but fido is asserted to be both - a genuine inconsistency
ERR = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")

g5 = StarLayerGraph()
g5.bind("ex", EX)
g5.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
""", format="turtle")

closed = g5.infer(profile="owl-rl")

# also_ns=ERR keeps the err:ErrorMessage/err:error triples visible alongside the
# ex: ones, even though their subject is a synthetic BNode rather than ex:something
show(closed, "After infer(profile='owl-rl') with inconstency.", also_ns=ERR)


After infer(profile='owl-rl') with inconstency.
@prefix err: <http://www.daml.org/2002/03/agents/agent-ont#> .
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:Cat owl:sameAs ex:Cat .

ex:Dog owl:disjointWith ex:Cat ;
    owl:sameAs ex:Dog .

ex:fido a ex:Dog, ex:Cat ;
    owl:sameAs ex:fido .

_:N651767b450804ea88c997713c1bb8e31 err:error "Disjoint classes http://example.org/Dog and http://example.org/Cat have a common individual http://example.org/fido" ;
    a err:ErrorMessage .



## 3. Delta and in-place mutation

Sections 1-2 above used `infer()`'s default mode `full` to create a new graph with the inferred triples. 

The examples below demonstrate the two additional modes, `delta` and `in-place`.

### 3.1 Materializing only the delta

`infer(mode="delta")` returns only the *newly*-entailed triples. 


In [7]:
# Same as example 1 above.  
# delta produces the triples that are added

g3 = StarLayerGraph()
g3.bind("ex", EX)
g3.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle12")

closure = g3.infer(profile="rdfs", mode="full")
delta = g3.infer(profile="rdfs", mode="delta")

print("original:", len(g3), "triples")
print("full     (mode='full', default):  ", len(closure), "triples - includes the originals")
print("delta    (mode='delta'):          ", len(delta), "triples - just what's new")
show(delta, "delta only")

# the original combined with delta reproduces the full graph exactly
print("\noriginal ∪ delta == full closure:", set(closure) == set(g3) | set(delta))

original: 2 triples
full     (mode='full', default):   12 triples - includes the originals
delta    (mode='delta'):           10 triples - just what's new

delta only
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource .

ex:alice a rdfs:Resource, ex:Employee .


original ∪ delta == full closure: True


### 3.2 Mutating in place

`mode="in-place"` adds the newly-entailed triples into the original graph.  

Note: if you mutate a graph and later delete or edit triples, entailed triples created from the original data will remain.

In [8]:
g4 = StarLayerGraph()
g4.bind("ex", EX)
g4.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

print("before:", len(g4), "triples")
g4.infer(profile="rdfs", mode="in-place")
print("after: ", len(g4), "triples")
show(g4, "g4 after mutating in place")

# Removing the fact that justified the entailment
# doesn't retract the entailment itself - infer(mode="in-place") only ever adds triples,
# it never re-checks earlier entailments when the graph changes.
g4.remove((EX.alice, RDF.type, EX.Manager))
print("\nafter removing 'alice a Manager':")
print("alice a Manager:", (EX.alice, RDF.type, EX.Manager) in g4, "(gone, as expected)")
print("alice a Employee:", (EX.alice, RDF.type, EX.Employee) in g4, "(still there - stale)")

before: 2 triples
after:  12 triples

g4 after mutating in place
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a rdfs:Resource, ex:Employee, ex:Manager .


after removing 'alice a Manager':
alice a Manager: False (gone, as expected)
alice a Employee: True (still there - stale)


## Where to go next

1. **[Getting Started](01-getting-started.ipynb)**
2. **[Graphs](02-graphs.ipynb)**
   - 2.b **Inferencing** — this guide.
3. **[SPARQL](03-sparql.ipynb)**
   - 3.a **[SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language.
   - 3.b **[SPARQL inferencing](03b-sparql-inferencing.ipynb)** — `.query(..., entailment="rdfs"|"owl-rl")`, a per-call choice between query-time rdfs:subClassOf rewrite and querying a live union of a graph with a cached `infer(mode="delta")` (section 3.1 above) - not a fresh, uncached materialization on every call.
4. **[SHACL shapes](04-shacl-shapes.ipynb)**
   - 4.b **[SHACL inference rules](04b-shacl-inference-rules.ipynb)** — SHACL rules including execution ordering, rule sets, provenance.
5. **Other**
   - 5.e **[OWL 2 DL reasoning with HermiT](05e-owl-dl-reasoning.ipynb)** — `infer(profile="owl-dl")`, genuine DL reasoning (disjunctive entailment, sound consistency checking) via `owlready2` + Java HermiT; needs a real JVM, not just a pip install.